In [1]:
import pandas as pd
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

# 1. Daten laden
df = pd.read_csv("../data/ab_ag.tsv", sep="\t")
df_sars_cov2 = pd.read_csv("../data cleanup/pdb_ids_sars_cov2.csv")

# 2. Heavy Chains für jede PDB-ID aus df_sars_cov2 ermitteln
heavy_chain = []
for pdb_id in df_sars_cov2["pdb"]:
    heavy_chain_id = df.loc[df["pdb"] == pdb_id, "Hchain"].values
    if len(heavy_chain_id) > 0:
        heavy_chain.append(heavy_chain_id[0])
    else:
        heavy_chain.append(None)

# Neue Spalte anhängen
df_sars_cov2["heavy_chain"] = heavy_chain

df_sars_cov2

# 3. Funktion zur CDR-H3-Extraktion mit übergebener heavy_chain_id
def get_cdr_h3_sequence(pdb_file, chain_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("pdb", pdb_file)

    cdr_h3_residues = []
    for model in structure:
        for chain in model:
            if chain.id == chain_id:
                for res in chain:
                    res_id = res.get_id()
                    if res_id[0] == " " and 95 <= res_id[1] <= 102:  # Chothia H3
                        cdr_h3_residues.append(res)

# Sequenz extrahieren
    aa_seq = ''.join(seq1(res["CA"].get_parent().resname) for res in cdr_h3_residues if "CA" in res)
    return aa_seq

cdr_h3_seqs = []

for idx, row in df_sars_cov2.iterrows():
    pdb_id = row["pdb"]
    chain_id = row["heavy_chain"]
    
    pdb_file_path = f"../data/pdb_cnf/{pdb_id}.pdb"

    
    if os.path.exists(pdb_file_path) and chain_id:
        try:
            cdr_h3_seq = get_cdr_h3_sequence(pdb_file_path, chain_id)
        except Exception as e:
            cdr_h3_seq = None
            print(f"Error for {pdb_id}: {e}")
    else:
        cdr_h3_seq = None

    cdr_h3_seqs.append(cdr_h3_seq)

    # 5. Als neue Spalte speichern und optional exportieren
df_sars_cov2["CDR_H3"] = cdr_h3_seqs
df_sars_cov2.to_csv("sars_cov2_cdrh3.tsv", sep="\t", index=False)

# Vorschau
print(df_sars_cov2.head())

df_sars_cov2


    pdb heavy_chain                 CDR_H3
0  9cci           B       TFGTYYDNTEDWFFDF
1  9ccj           H              LPLGERIDY
2  9bj2           H  HNGDPYDFWSGYNTWAGGLDV
3  8z6r           E           QGDLGDWILLGY
4  8z6s           E           QGDLGDWILLGY


,pdb,heavy_chain,CDR_H3
0,9cci,B,TFGTYYDNTEDWFFDF
1,9ccj,H,LPLGERIDY
2,9bj2,H,HNGDPYDFWSGYNTWAGGLDV
3,8z6r,E,QGDLGDWILLGY
4,8z6s,E,QGDLGDWILLGY
...,...,...,...
374,7tly,A,DYTRGAWFGESLIGGFDN
375,7chp,H,DLQEHGMDV
376,7n4i,H,LLYYSDSSPLDS
377,7l7d,H,PYCSSISCNDGFDI
